# Predicting US Flight Delays — linear pipeline

**Dataset:** USDOT 2015 Flight Delays (Kaggle), stored in HDFS (local Docker cluster).
**Target:** `is_delayed = 1` when `ARRIVAL_DELAY >= 15` minutes (BTS definition: *15 minutes or more*).
**Question:** can we predict an arrival delay **before departure**, using only the flight schedule?

This notebook runs top to bottom in a fresh kernel:

1. Configuration & Spark session
2. Load the CSV from HDFS with an explicit schema, convert it once to Parquet
3. Exploration: missing values, target, correlations, data leakage, known anomalies
4. Cleaning: airport codes, cancelled/diverted flights, duplicates, label
5. Train / validation / test split
6. Feature engineering (fitted on the training set only)
7. Models: baseline, Logistic Regression, Random Forest, Gradient-Boosted Trees, with class weights
8. Decision threshold tuned on the validation set, final scores on the test set
9. Export small CSV files for the Streamlit app

## 1) Configuration & Spark session
Every path and setting lives here. `HDFS_URL` points to the NameNode of the local Docker cluster.
Set `SAMPLE_FRACTION` to `0.1` while developing, `1.0` for the final run.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, types as T

HDFS_URL       = os.environ.get("HDFS_URL", "hdfs://localhost:9000")
RAW_CSV        = f"{HDFS_URL}/data/raw/flights.csv"
AIRPORT_ID_CSV = f"{HDFS_URL}/data/raw/L_AIRPORT_ID.csv"   # BTS lookup: 5-digit airport ID -> description
AIRPORT_CSV    = f"{HDFS_URL}/data/raw/L_AIRPORT.csv"      # BTS lookup: IATA code -> description
PARQUET        = f"{HDFS_URL}/data/processed/flights.parquet"

SAMPLE_FRACTION = float(os.environ.get("SAMPLE_FRACTION", "1.0"))
SEED            = 42
DELAY_THRESHOLD = 15    # minutes, BTS definition of a delayed flight
TOP_AIRPORTS    = 30    # airports kept by name, the rest grouped as "OTHER"

# outputs/ at the repo root, whether the notebook runs from the root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs"

spark = (SparkSession.builder.appName("flight-delays-linear")
         .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def path_exists(path: str) -> bool:
    # True if the file or folder exists on HDFS (or any Hadoop filesystem)
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    return p.getFileSystem(spark._jsc.hadoopConfiguration()).exists(p)

print("Spark", spark.version, "| HDFS:", HDFS_URL, "| sample:", SAMPLE_FRACTION)

## 2) Load the data

### 2.1) Explicit schema
Instead of reading everything as text and casting 26 columns afterwards, we give Spark the schema up front.
Times such as `0005` are read as the integer `5` (hhmm format). Empty fields become `null`.

In [ ]:
INT, DBL, STR = T.IntegerType(), T.DoubleType(), T.StringType()
schema = T.StructType([T.StructField(name, typ) for name, typ in [
    ("YEAR", INT), ("MONTH", INT), ("DAY", INT), ("DAY_OF_WEEK", INT), ("AIRLINE", STR),
    ("FLIGHT_NUMBER", INT), ("TAIL_NUMBER", STR), ("ORIGIN_AIRPORT", STR), ("DESTINATION_AIRPORT", STR),
    ("SCHEDULED_DEPARTURE", INT), ("DEPARTURE_TIME", INT), ("DEPARTURE_DELAY", DBL), ("TAXI_OUT", DBL),
    ("WHEELS_OFF", INT), ("SCHEDULED_TIME", DBL), ("ELAPSED_TIME", DBL), ("AIR_TIME", DBL), ("DISTANCE", DBL),
    ("WHEELS_ON", INT), ("TAXI_IN", DBL), ("SCHEDULED_ARRIVAL", INT), ("ARRIVAL_TIME", INT), ("ARRIVAL_DELAY", DBL),
    ("DIVERTED", INT), ("CANCELLED", INT), ("CANCELLATION_REASON", STR),
    ("AIR_SYSTEM_DELAY", DBL), ("SECURITY_DELAY", DBL), ("AIRLINE_DELAY", DBL),
    ("LATE_AIRCRAFT_DELAY", DBL), ("WEATHER_DELAY", DBL),
]])

### 2.2) CSV → Parquet (once)
The raw CSV stays untouched in `/data/raw/`. The first run writes a typed Parquet copy to `/data/processed/`;
later runs read the Parquet directly, which is much faster (columnar, compressed, only the needed columns are read).

In [ ]:
if not path_exists(PARQUET):
    (spark.read.option("header", "true").schema(schema).csv(RAW_CSV)
          .write.mode("overwrite").parquet(PARQUET))
    print("Parquet written to", PARQUET)

df = spark.read.parquet(PARQUET)
n_raw = df.count()
print(f"Rows: {n_raw:,} | Columns: {len(df.columns)}")

The real file has **5,819,079 rows and 31 columns**, the number Kaggle announces.

In [ ]:
pd.set_option("display.max_columns", None)
df.limit(5).toPandas()

## 3) Exploration
We explore a 5% random sample, cached in memory. Cleaning and training use the full table.

In [ ]:
eda = df.sample(fraction=0.05, seed=SEED).persist(StorageLevel.MEMORY_AND_DISK)
print(f"EDA sample: {eda.count():,} rows")

### 3.1) Missing values

In [ ]:
missing = eda.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in eda.columns]).first().asDict()
(pd.Series(missing, name="missing_%") * 100).round(2).sort_values(ascending=False).to_frame().T

- Every column known **before departure** (airline, airports, schedule, distance) is complete.
- The 5 delay-cause columns are only filled for delayed flights.
- Arrival columns are missing for a small share of flights. We check below that these are exactly the cancelled or diverted ones.

In [ ]:
(eda.groupBy("CANCELLED", "DIVERTED")
    .agg(F.count("*").alias("flights"),
         F.sum(F.col("ARRIVAL_DELAY").isNull().cast("int")).alias("missing_arrival_delay"))
    .orderBy("CANCELLED", "DIVERTED").toPandas())

Every missing `ARRIVAL_DELAY` belongs to a cancelled or diverted flight: they have no arrival, so we remove them.

### 3.2) Target distribution
A flight is **delayed** when it arrives **15 minutes or more** late (BTS definition).

In [ ]:
flown = (eda.filter(F.col("ARRIVAL_DELAY").isNotNull())
            .withColumn("is_delayed", (F.col("ARRIVAL_DELAY") >= DELAY_THRESHOLD).cast("int")))
target = flown.groupBy("is_delayed").count().orderBy("is_delayed").toPandas()

plt.figure(figsize=(4.5, 4.5))
plt.pie(target["count"], autopct="%.1f%%", startangle=90,
        labels=[f"{'Delayed ≥15 min' if d else 'Not delayed'}\n({n:,})" for d, n in zip(target.is_delayed, target["count"])])
plt.title("Target distribution (is_delayed)")
plt.show()

The classes are imbalanced (about 4 flights out of 5 are not delayed). A model that always answers *not delayed*
already scores about 82% accuracy. Accuracy alone is therefore meaningless here: we will judge the models on the
**delayed** class (precision, recall, F1) and on the area under the precision-recall curve.

### 3.3) Correlations with the target
Pearson correlation between each numeric column and `is_delayed`. Columns measured **after** departure are shown
on purpose: they show which information would leak into a pre-departure model.

In [ ]:
corr_cols = ["MONTH", "DAY", "DAY_OF_WEEK", "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE",
             "DEPARTURE_DELAY", "TAXI_OUT", "TAXI_IN", "AIR_TIME", "ELAPSED_TIME"]
correlations = flown.select([F.corr(F.col(c).cast("double"), F.col("is_delayed").cast("double")).alias(c)
                             for c in corr_cols]).first().asDict()
corr_series = pd.Series(correlations).sort_values()

post_departure = {"DEPARTURE_DELAY", "TAXI_OUT", "TAXI_IN", "AIR_TIME", "ELAPSED_TIME"}
colors = ["tab:red" if c in post_departure else "tab:blue" for c in corr_series.index]
plt.figure(figsize=(7, 4))
plt.barh(corr_series.index, corr_series.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlation with is_delayed  (red = known only after departure)")
plt.tight_layout()
plt.show()

- `DEPARTURE_DELAY` and `TAXI_OUT` are the strongest signals, but they are only known **after** departure: using them would be data leakage.
- Among pre-departure columns, `SCHEDULED_DEPARTURE` (time of day) has the clearest link: delays build up during the day.
- `MONTH` and `DAY_OF_WEEK` show almost no *linear* correlation. That does not mean they have no effect: it is
  seasonal and not linear (e.g. summer and December peaks), so we will one-hot encode them instead of treating them as numbers.

### 3.4) Proof of leakage in the delay-cause columns
The documentation says that the 5 delay-cause columns are filled only for delayed flights and that they add up
to the arrival delay. We check it on the data before dropping them.

In [ ]:
causes = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
with_causes = (eda.filter(F.col("AIRLINE_DELAY").isNotNull())
                  .withColumn("causes_sum", sum(F.col(c) for c in causes)))
with_causes.agg(
    F.count("*").alias("flights_with_causes"),
    F.avg((F.col("causes_sum") == F.col("ARRIVAL_DELAY")).cast("int")).alias("share_sum_equals_arrival_delay"),
    F.min("ARRIVAL_DELAY").alias("min_arrival_delay"),
).toPandas()

The causes add up to `ARRIVAL_DELAY` and only exist when the flight is at least 15 minutes late:
they are the answer itself and are excluded. The same goes for every column measured after departure.

| Kept (known before departure) | Dropped |
|---|---|
| `MONTH`, `DAY_OF_WEEK`, `AIRLINE`, `ORIGIN_AIRPORT`, `DESTINATION_AIRPORT`, `SCHEDULED_DEPARTURE`, `SCHEDULED_TIME`, `DISTANCE` | `DEPARTURE_TIME`, `DEPARTURE_DELAY`, `TAXI_OUT`, `WHEELS_OFF`, `WHEELS_ON`, `TAXI_IN`, `ARRIVAL_TIME`, `ELAPSED_TIME`, `AIR_TIME`, `CANCELLATION_REASON`, the 5 delay causes |

`TAIL_NUMBER`, `FLIGHT_NUMBER` and `DAY` are left out (identifiers or too many values to generalise);
`SCHEDULED_ARRIVAL` is redundant with departure time + scheduled duration.
`ARRIVAL_DELAY`, `CANCELLED` and `DIVERTED` are used only to filter flights and build the label.

### 3.5) Known anomalies
Two quirks of the 2015 file are documented on Kaggle:
1. Airports written as **5-digit BTS IDs** instead of 3-letter IATA codes.
2. Midnight written as **2400**, which would give an hour 24.

In [ ]:
numeric_code = F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") | F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$")
(eda.groupBy("MONTH")
    .agg(F.count("*").alias("flights"), F.sum(numeric_code.cast("int")).alias("numeric_airport_codes"))
    .orderBy("MONTH").toPandas().T)

In [ ]:
print("SCHEDULED_DEPARTURE == 2400 in the full table:", df.filter(F.col("SCHEDULED_DEPARTURE") == 2400).count())

- The numeric codes are concentrated in **October**. Dropping them would remove a whole month from training,
  and the model would never learn October. Instead we translate them back to IATA codes with the official BTS lookup tables
  (`L_AIRPORT_ID` gives ID → airport name, `L_AIRPORT` gives IATA code → airport name, both from BTS TranStats),
  stored next to the raw file in `/data/raw/`. If these files are missing, the notebook falls back to dropping those rows and says so.
- `2400` is handled anyway when we compute the departure hour (`% 2400`), so midnight becomes hour 0.

In [ ]:
eda.unpersist();

## 4) Clean the data (full table)

### 4.1) Translate numeric airport codes back to IATA

In [ ]:
def fix_numeric_airports(frame):
    if not (path_exists(AIRPORT_ID_CSV) and path_exists(AIRPORT_CSV)):
        print("BTS lookup tables not found in /data/raw/: rows with numeric airport codes will be dropped (≈ October).")
        return frame

    ids  = spark.read.option("header", "true").csv(AIRPORT_ID_CSV).toDF("ID", "DESCRIPTION")
    iata = spark.read.option("header", "true").csv(AIRPORT_CSV).toDF("IATA", "DESCRIPTION")

    # IATA codes really used in 2015 (outside October): resolves names shared by several codes
    used = (frame.select(F.col("ORIGIN_AIRPORT").alias("IATA"))
                 .union(frame.select(F.col("DESTINATION_AIRPORT").alias("IATA")))
                 .filter(~F.col("IATA").rlike("^[0-9]+$")).distinct())
    mapping = F.broadcast(ids.join(iata, "DESCRIPTION").join(used, "IATA")
                             .groupBy("ID").agg(F.min("IATA").alias("IATA")))

    for col in ["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]:
        m = mapping.withColumnRenamed("ID", "_id").withColumnRenamed("IATA", "_iata")
        frame = (frame.join(m, F.col(col) == F.col("_id"), "left")
                      .withColumn(col, F.coalesce(F.col("_iata"), F.col(col)))
                      .drop("_id", "_iata"))
    return frame

flights = fix_numeric_airports(df)
still_numeric = flights.filter(F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") | F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$")).count()
print(f"Rows still with a numeric airport code: {still_numeric:,}")

### 4.2) Filter, deduplicate, build the label
Each step logs how many rows it removes.

- **Cancelled / diverted flights**: no arrival, so no label.
- **Missing or impossible values** in the columns we use.
- **Duplicates** on the flight key (date, airline, flight number, origin, scheduled departure). We do not deduplicate
  on the reduced set of model columns: two different real flights (e.g. the same weekly flight on two days of the same month)
  can share all those values and would be merged by mistake.

In [ ]:
KEY = ["YEAR", "MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "SCHEDULED_DEPARTURE"]
USED = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE", "ARRIVAL_DELAY"]

log = [("raw", n_raw)]
step = flights.filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))
log.append(("without cancelled / diverted", step.count()))

step = step.dropna(subset=USED)
log.append(("without missing values", step.count()))

step = step.filter(~F.col("ORIGIN_AIRPORT").rlike("^[0-9]+$") & ~F.col("DESTINATION_AIRPORT").rlike("^[0-9]+$")
                   & (F.col("SCHEDULED_TIME") > 0) & (F.col("DISTANCE") > 0))
log.append(("without numeric codes / impossible values", step.count()))

step = step.dropDuplicates(KEY)
log.append(("without duplicates (flight key)", step.count()))

steps = pd.DataFrame(log, columns=["step", "rows"])
steps["removed"] = (-steps["rows"].diff()).fillna(0).astype(int)
steps

In [ ]:
clean = (step
    .withColumn("dep_hour", F.floor((F.col("SCHEDULED_DEPARTURE") % 2400) / 100).cast("int"))   # 2400 -> hour 0
    .withColumn("label", (F.col("ARRIVAL_DELAY") >= DELAY_THRESHOLD).cast("int"))
    .select("MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "dep_hour",
            "SCHEDULED_TIME", "DISTANCE", "ARRIVAL_DELAY", "label"))

if SAMPLE_FRACTION < 1.0:
    clean = clean.sample(fraction=SAMPLE_FRACTION, seed=SEED)

clean = clean.persist(StorageLevel.MEMORY_AND_DISK)
n_clean = clean.count()
print(f"Clean rows used for modelling: {n_clean:,}")
clean.groupBy("label").count().orderBy("label").toPandas()

## 5) Train / validation / test split
- **Train (70%)**: the models learn on it.
- **Validation (15%)**: used only to choose the decision threshold.
- **Test (15%)**: touched once, at the end, for the final scores.

In [ ]:
train, val, test = clean.randomSplit([0.70, 0.15, 0.15], seed=SEED)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    n, pos = part.agg(F.count("*"), F.avg("label")).first()
    print(f"{name:<11} {n:>10,} rows | delayed: {pos:.1%}")

## 6) Feature engineering
Everything is learned on the **training set only**, then applied to validation and test.

- **Airports**: the `TOP_AIRPORTS` busiest origins and destinations keep their name, the others become `OTHER`.
- **Categories → one-hot vectors**: airline, origin, destination, and also month, day of week and departure hour
  (their effect is not linear, so a single number would mislead the Logistic Regression).
- **Numbers kept as is**: scheduled duration and distance.
- **Class weights**: delayed flights are rarer, so each one weighs more during training. Without this,
  the models simply learn to answer *not delayed* every time.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, OneHotEncoder, VectorAssembler

top_origins = [r[0] for r in train.groupBy("ORIGIN_AIRPORT").count().orderBy(F.desc("count")).limit(TOP_AIRPORTS).collect()]
top_dests   = [r[0] for r in train.groupBy("DESTINATION_AIRPORT").count().orderBy(F.desc("count")).limit(TOP_AIRPORTS).collect()]
sql_list = lambda codes: ", ".join(f"'{c}'" for c in codes)

# airport grouping as a pipeline stage, so the saved model can be reused as is
bucketizer = SQLTransformer(statement=(
    "SELECT *, "
    f"CASE WHEN ORIGIN_AIRPORT IN ({sql_list(top_origins)}) THEN ORIGIN_AIRPORT ELSE 'OTHER' END AS origin_bucket, "
    f"CASE WHEN DESTINATION_AIRPORT IN ({sql_list(top_dests)}) THEN DESTINATION_AIRPORT ELSE 'OTHER' END AS dest_bucket "
    "FROM __THIS__"))

cat_text = ["AIRLINE", "origin_bucket", "dest_bucket"]
cat_num  = ["MONTH", "DAY_OF_WEEK", "dep_hour"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_text]
encoder  = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_text] + cat_num,
                         outputCols=[f"{c}_vec" for c in cat_text + cat_num], handleInvalid="keep")
assembler = VectorAssembler(inputCols=[f"{c}_vec" for c in cat_text + cat_num] + ["SCHEDULED_TIME", "DISTANCE"],
                            outputCol="features")
feature_model = Pipeline(stages=[bucketizer] + indexers + [encoder, assembler]).fit(train)

# balanced class weights, computed on train
pos_rate = train.agg(F.avg("label")).first()[0]
weight = F.when(F.col("label") == 1, 0.5 / pos_rate).otherwise(0.5 / (1 - pos_rate))

def featurize(frame):
    return feature_model.transform(frame).select("features", "label", weight.alias("weight"))

train_f = featurize(train).persist(StorageLevel.MEMORY_AND_DISK)
val_f   = featurize(val).persist(StorageLevel.MEMORY_AND_DISK)
test_f  = featurize(test).persist(StorageLevel.MEMORY_AND_DISK)
print("Features per flight:", train_f.first()["features"].size)
print(f"Weights -> delayed: {0.5 / pos_rate:.2f} | not delayed: {0.5 / (1 - pos_rate):.2f}")

## 7) Models
- **Baseline**: always answers *not delayed*. Any useful model must beat it on the delayed class.
- **Logistic Regression**, **Random Forest**, **Gradient-Boosted Trees**, all trained with the class weights.

GBT is the slowest. Remove it from `models` if training takes too long on your machine.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

models = {
    "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="label", weightCol="weight",
                                             maxIter=50, regParam=0.001),
    "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight",
                                           numTrees=50, maxDepth=10, seed=SEED),
    "GBT": GBTClassifier(featuresCol="features", labelCol="label", weightCol="weight",
                         maxIter=40, maxDepth=6, seed=SEED),
}

def scores(model, frame):
    # label + probability of 'delayed'
    return model.transform(frame).select("label", vector_to_array("probability")[1].alias("p1"))

THRESHOLDS = [round(t, 3) for t in np.arange(0.05, 0.951, 0.025)]

def threshold_curve(sc):
    # precision / recall / F1 of the delayed class for every threshold, in a single Spark pass
    aggs = [F.sum("label").alias("pos")]
    for i, t in enumerate(THRESHOLDS):
        hit = F.col("p1") >= t
        aggs += [F.sum((hit & (F.col("label") == 1)).cast("long")).alias(f"tp{i}"),
                 F.sum((hit & (F.col("label") == 0)).cast("long")).alias(f"fp{i}")]
    r = sc.agg(*aggs).first()
    rows = []
    for i, t in enumerate(THRESHOLDS):
        tp, fp = r[f"tp{i}"], r[f"fp{i}"]
        fn = r["pos"] - tp
        precision = tp / (tp + fp) if tp + fp else np.nan   # undefined when nothing is predicted delayed
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if tp else 0.0
        rows.append((t, precision, recall, f1))
    return pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])

auc_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="p1", metricName="areaUnderROC")
auc_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="p1", metricName="areaUnderPR")

def evaluate(sc, threshold):
    pred = sc.withColumn("pred", (F.col("p1") >= threshold).cast("int"))
    cm = {(r["label"], r["pred"]): r["count"] for r in pred.groupBy("label", "pred").count().collect()}
    tp, fp = cm.get((1, 1), 0), cm.get((0, 1), 0)
    fn, tn = cm.get((1, 0), 0), cm.get((0, 0), 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "threshold": threshold,
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "precision_delayed": precision,
        "recall_delayed": recall,
        "f1_delayed": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "auc_roc": auc_roc.evaluate(sc),
        "auc_pr": auc_pr.evaluate(sc),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

### 7.1) Train, tune the threshold on validation, score on test
A model outputs a probability of delay. The usual cut at 0.5 is arbitrary, especially with imbalanced classes:
we pick, **on the validation set**, the threshold that maximises the F1 score of the delayed class, then apply it unchanged to the test set.

In [ ]:
import time

results, curves, fitted = [], {}, {}

# baseline: 'not delayed' for everyone (constant probability 0)
base = test_f.select("label", F.lit(0.0).alias("p1"))
results.append({"model": "Baseline (always not delayed)", **evaluate(base, 0.5)})

for name, clf in models.items():
    t0 = time.time()
    model = clf.fit(train_f)
    fitted[name] = model
    curve = threshold_curve(scores(model, val_f))
    best_t = float(curve.loc[curve["f1"].idxmax(), "threshold"])
    curves[name] = curve
    res = {"model": name, **evaluate(scores(model, test_f), best_t)}
    results.append(res)
    print(f"{name:<19} {time.time() - t0:6.0f}s | threshold {best_t:.2f} | "
          f"F1 delayed {res['f1_delayed']:.3f} | AUC-PR {res['auc_pr']:.3f} | AUC-ROC {res['auc_roc']:.3f}")

scores_df = pd.DataFrame(results)
scores_df[["model", "threshold", "accuracy", "precision_delayed", "recall_delayed", "f1_delayed", "auc_roc", "auc_pr"]].round(3)

How to read this table:
- The **baseline** has a high accuracy but finds **no** delayed flight (recall 0). This is what the previous version of the notebook
  was actually measuring: its models predicted *not delayed* for every flight.
- **AUC-PR** is the main comparison metric here. A random model scores the share of delayed flights (about 0.18 on the real data). Higher is better.
- **Recall delayed** answers *"what share of the real delays do we catch?"*, **precision delayed** answers *"when we announce a delay, how often is it real?"*.

### 7.2) Best model: threshold curve and confusion matrix

In [ ]:
best = scores_df.iloc[1:].sort_values("auc_pr", ascending=False).iloc[0]
best_name = best["model"]
curve = curves[best_name]

plt.figure(figsize=(6.5, 3.8))
for metric in ["precision", "recall", "f1"]:
    plt.plot(curve["threshold"], curve[metric], label=metric)
plt.axvline(best["threshold"], color="grey", linestyle="--", linewidth=0.8)
plt.xlabel("Decision threshold (validation set)")
plt.title(f"{best_name}: delayed class")
plt.legend()
plt.tight_layout()
plt.show()

pd.DataFrame([[best["tn"], best["fp"]], [best["fn"], best["tp"]]],
             index=["actual: not delayed", "actual: delayed"],
             columns=["predicted: not delayed", "predicted: delayed"])

### 7.3) What drives the predictions
Feature importances of the best tree model, summed per original column (one-hot columns grouped back together).

In [ ]:
tree_name = next(n for n in scores_df.sort_values("auc_pr", ascending=False)["model"] if n in ("RandomForest", "GBT"))
attrs = train_f.schema["features"].metadata["ml_attr"]["attrs"]
names = {a["idx"]: a["name"] for group in attrs.values() for a in group}

def source_column(feature_name):
    for c in cat_text + cat_num:
        if feature_name.startswith(f"{c}_vec"):
            return c
    return feature_name

imp = fitted[tree_name].featureImportances.toArray()
importance = (pd.DataFrame({"feature": [source_column(names[i]) for i in range(len(imp))], "importance": imp})
                .groupby("feature", as_index=False)["importance"].sum()
                .sort_values("importance", ascending=True))

plt.figure(figsize=(6, 3.5))
plt.barh(importance["feature"], importance["importance"])
plt.title(f"Feature importance ({tree_name})")
plt.tight_layout()
plt.show()

## 8) Export for Streamlit
Small CSV files in `outputs/` at the repo root. The Streamlit app reads them and needs neither Spark nor HDFS.
`model_scores.csv` keeps the columns of the previous version (`model`, `accuracy`, `f1`, `auc`) and adds the new ones.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def group_stats(col):
    return (clean.groupBy(col)
                 .agg(F.count("*").alias("flights"),
                      F.round(F.avg("ARRIVAL_DELAY"), 2).alias("avg_arr_delay"),
                      F.round(F.avg("label") * 100, 1).alias("delay_rate_pct"))
                 .orderBy(col).toPandas())

group_stats("AIRLINE").sort_values("avg_arr_delay", ascending=False).to_csv(OUTPUT_DIR / "stats_airline.csv", index=False)
group_stats("dep_hour").to_csv(OUTPUT_DIR / "stats_hour.csv", index=False)
group_stats("MONTH").to_csv(OUTPUT_DIR / "stats_month.csv", index=False)

export = scores_df.assign(f1=scores_df["f1_delayed"], auc=scores_df["auc_roc"])
export.to_csv(OUTPUT_DIR / "model_scores.csv", index=False)
curves[best_name].assign(model=best_name).to_csv(OUTPUT_DIR / "threshold_curve.csv", index=False)
importance.sort_values("importance", ascending=False).to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

print("Written to", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))

## 9) (Optional) Save the best model to HDFS
Feature steps + model in a single `PipelineModel`, reloadable with `PipelineModel.load(path)`.

In [ ]:
from pyspark.ml import PipelineModel

SAVE_MODEL = False
if SAVE_MODEL:
    path = f"{HDFS_URL}/models/flight_delays_{best_name}"
    PipelineModel(stages=feature_model.stages + [fitted[best_name]]).write().overwrite().save(path)
    print("Model saved to", path, "| threshold:", best["threshold"])